In [2]:
from parity import calculate_fourier_transform_matrix, popcount
import numpy as np
from misc_utils import make_unpacked_configurations, make_packed_configurations
import matplotlib.pyplot as plt
import numpy.typing as npt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data.dataset import random_split
from torch.optim import Adam
from torch.nn import functional as F
# binomial coefficients:
from scipy.special import comb
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from nn_xors_2023_07_18 import FourierSeries, similar_distance_network

In [15]:
make_unpacked_configurations(5, 5)

array([1, 0, 1, 0, 0], dtype=uint64)

In [16]:
n_spins = 24

In [35]:
assert np.allclose(
    FourierSeries(xors=np.array([1, 6]), coeffs=np.array([3, 7]))(
        np.array([0, 1, 2, 3, 4, 5, 6, 7])
    ),
    np.array([10.0, 4.0, -4.0, -10.0, -4.0, -10.0, 10.0, 4.0]),
)

In [18]:
xors = make_packed_configurations(np.random.randint(0, 2, size=(1000, n_spins)), n_spins)

In [25]:
n_spins = 24
while True:
    all_xors = np.arange(2**n_spins, dtype=np.uint64)
    all_xors = all_xors[popcount(all_xors) == 10]
    new_xors = similar_distance_network(all_xors, 2, 4, 18)
    if len(new_xors) == 18:
        break
# popcount(new_xors.reshape(-1, 1) ^ new_xors.reshape(1, -1))

In [21]:
# popcount(new_xors.reshape(-1, 1) ^ new_xors.reshape(1, -1))

In [45]:
(16 - 12) / 2

2.0

In [47]:
hamming_weight = 8

all_xors = np.arange(2**n_spins, dtype=np.uint64)
all_xors = all_xors[popcount(all_xors) == hamming_weight]

distance = 16
n_xors = 3
len(similar_distance_network(
    all_xors, distance_min=distance, distance_max=distance + 2, maxitems=n_xors
))

3

In [26]:
n_spins = 24
eps_test = 0.03
all_states = np.arange(2**n_spins, dtype=np.uint64)
all_states = all_states[popcount(all_states) == n_spins // 2]
eps_train = 0.003 * comb(n_spins, n_spins // 2) / len(all_states)
batch_size = 64
n_xors = 2
n_hidden = 512
runs = 5
for distance in [18]:
    for run in range(runs):
        writer = SummaryWriter(
            log_dir=(
                f"experiments/2023_07_18/{datetime.now().strftime('%H_%M_%S')}"
                f"_{n_xors=}_{distance=}_{n_hidden=}_{eps_train=}"
            )
        )

        sample_states = np.random.choice(
            all_states,
            size=int(len(all_states) * (eps_train + eps_test)),
            replace=False,
        )
        hamming_weight = 10

        all_xors = np.arange(2**n_spins, dtype=np.uint64)
        all_xors = all_xors[popcount(all_xors) == hamming_weight]

        for _ in range(2):
            selected_xors = similar_distance_network(
                all_xors, distance_min=distance, distance_max=distance + 2, maxitems=n_xors
            )
            if len(selected_xors) == n_xors:
                break
        else:
            raise ValueError("Could not find enough xors")

        series = FourierSeries(selected_xors, 2 * np.random.rand(len(selected_xors)) - 1)

        dataset = make_dataset(series, sample_states)
        train_size = int(0.8 * len(dataset))
        test_size = len(dataset) - train_size
        train_dataset, test_dataset = random_split(
            dataset, [eps_train / (eps_train + eps_test), eps_test / (eps_train + eps_test)]
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

        net = MLPBinaryClassifier(n_spins, n_hidden)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

        train(
            net,
            criterion,
            optimizer,
            train_loader,
            test_dataset=test_dataset,
            n_epochs=500,
            writer=writer,
            break_on_loss=1e-04,
        )

Epoch	0	loss	0.6898	accuracy	0.5510	Accuracy (test)	0.5021
Epoch	1	loss	0.6875	accuracy	0.5510	Accuracy (test)	0.5004
Epoch	2	loss	0.6905	accuracy	0.5918	Accuracy (test)	0.5025
Epoch	3	loss	0.6615	accuracy	0.6122	Accuracy (test)	0.5019
Epoch	4	loss	0.7102	accuracy	0.4082	Accuracy (test)	0.4993
Epoch	5	loss	0.6281	accuracy	0.6939	Accuracy (test)	0.4999
Epoch	6	loss	0.7055	accuracy	0.4694	Accuracy (test)	0.5041
Epoch	7	loss	0.6593	accuracy	0.4898	Accuracy (test)	0.4998
Epoch	8	loss	0.6726	accuracy	0.6122	Accuracy (test)	0.5013
Epoch	9	loss	0.6476	accuracy	0.5714	Accuracy (test)	0.5027
Epoch	10	loss	0.6660	accuracy	0.5714	Accuracy (test)	0.5035
Epoch	11	loss	0.6371	accuracy	0.6531	Accuracy (test)	0.5025
Epoch	12	loss	0.6474	accuracy	0.5510	Accuracy (test)	0.5032
Epoch	13	loss	0.6410	accuracy	0.6327	Accuracy (test)	0.5017
Epoch	14	loss	0.6120	accuracy	0.6531	Accuracy (test)	0.4991
Epoch	15	loss	0.6621	accuracy	0.5918	Accuracy (test)	0.5040
Epoch	16	loss	0.6158	accuracy	0.6327	Accuracy (tes

KeyboardInterrupt: 

In [ ]:
all_xors = all_xors[popcount(all_xors) == hamming_weight]
sample_xors = np.random.choice(all_xors, size=1000, replace=False)
plt.hist(popcount(sample_xors.reshape(-1, 1) ^ sample_xors.reshape(1, -1)).reshape(-1), bins=20)

NameError: name 'hamming_weight' is not defined

In [ ]:
distance = 14
for _ in range(1):
    selected_xors = similar_distance_network(
        all_xors, distance_min=distance, distance_max=distance + 2, maxitems=n_xors
    )
    if len(selected_xors) == n_xors:
        break
else:
    raise ValueError("Could not find enough xors")

ValueError: Could not find enough xors

In [ ]:
len(selected_xors)

5